# Parallel And Larger-Than-Memory Workflows

Raw binary outputs from simulations are often larger than the memory of a single machine. xarray-binfile builds on Dask, so the same code that reads a handful of files also scales to collections that do not fit in RAM.

This notebook shows how to:

- open many files at once with `open_mfdataset(..., parallel=True)`
- treat chunks as the unit of parallel, out-of-core work
- choose a Dask scheduler and worker count for a compute step
- evaluate several lazy results together with `dask.compute`
- reduce data larger than memory without loading it all
- stream a derived result back to disk one file at a time

```{note}
The dataset here is deliberately tiny so the documentation builds quickly. The important part is the *pattern*: because Dask keeps only a few chunks in memory at a time, the identical code scales to datasets much larger than RAM.
```

In [ ]:
import pathlib
import tempfile

import dask
import numpy as np
import xarray as xr

import xarray_binfile  # noqa: F401  (registers the .binary_engine accessors)
from xarray_binfile.tutorial import DatasetGenerator, FileSpecsGetter

source_directory_holder = tempfile.TemporaryDirectory()
derived_directory_holder = tempfile.TemporaryDirectory()
source_directory = pathlib.Path(source_directory_holder.name)
derived_directory = pathlib.Path(derived_directory_holder.name)
source_directory

## Generate example data

The source data uses `float32` values on a small 3D grid, stored as one file per variable and timestep. This mirrors a typical simulation output directory with `ux`, `uy`, and `uz` velocity components.

In [ ]:
base_coords = {
    "x": np.linspace(0.0, 3.0, num=32, dtype=np.float32),
    "y": np.linspace(-1.0, 1.0, num=24, dtype=np.float32),
    "z": np.linspace(0.0, 2.0, num=16, dtype=np.float32),
}

getter = FileSpecsGetter(base_coords=base_coords, dtype=np.float32)
dataset_generator = DatasetGenerator(getter.reader)
filenames = (
    getter.filename_template.format(name=name, digits=step)
    for name in ("ux", "uy", "uz")
    for step in range(12)
)
source_dataset = dataset_generator(map(pathlib.Path, filenames))
source_dataset.binary_engine.to_file(getter.writer, source_directory)
sorted(path.name for path in source_directory.glob("*.bin"))[:6]

## Open many files in parallel

`xr.open_mfdataset` reads and combines the files by coordinate. Passing `parallel=True` opens the files concurrently with `dask.delayed`, and passing `chunks` keeps every variable lazy and Dask-backed instead of loading it eagerly.

In [ ]:
lazy_dataset = xr.open_mfdataset(
    sorted(source_directory.glob("*.bin")),
    engine="binfile",
    read_specs_getter=getter.reader,
    chunks={"x": 8, "y": 6, "z": 4, "time": 3},
    parallel=True,
)
lazy_dataset

## Chunks are the unit of parallel work

Each chunk is an independent task that Dask can hand to a different worker, so more chunks means more opportunities for parallelism. Chunking is also what bounds memory use: only the chunks currently in flight are resident, no matter how large the full array is on disk.

In [ ]:
ux = lazy_dataset["ux"]
blocks_per_dim = dict(zip(ux.dims, ux.data.numblocks))
largest_chunk_bytes = (
    int(np.prod([max(sizes) for sizes in ux.data.chunks])) * ux.dtype.itemsize
)

print("array type:", type(ux.data).__name__)
print("blocks per dim:", blocks_per_dim)
print("independent chunks (parallel tasks):", ux.data.npartitions)
print(f"whole ux array: {ux.nbytes / 1024:.1f} KiB")
print(f"largest single chunk: {largest_chunk_bytes / 1024:.1f} KiB")

## Choose a scheduler for a compute step

Dask arrays default to the threaded scheduler, which suits the NumPy-heavy reads this backend performs. You can set the scheduler and worker count for a specific computation with `dask.config.set`. When running interactively, you can also wrap the call in Dask's [`ProgressBar`](https://docs.dask.org/en/stable/diagnostics-local.html) context manager to watch the chunks complete in parallel.

For CPU-bound pure-Python work use `scheduler='processes'`, and for multi-machine clusters install [`dask.distributed`](https://distributed.dask.org) and create a `Client`. The analysis code stays the same.

In [ ]:
speed = np.sqrt(
    lazy_dataset["ux"] ** 2 + lazy_dataset["uy"] ** 2 + lazy_dataset["uz"] ** 2
).rename("speed")
lazy_mean_speed = speed.mean("time")

with dask.config.set(scheduler="threads", num_workers=4):
    mean_speed = lazy_mean_speed.compute()

mean_speed

## Evaluate several results together

When you need more than one result, pass them all to `dask.compute` in a single call. Dask then builds one combined task graph, reads each shared chunk only once, and computes the outputs in parallel instead of walking the data multiple times.

In [ ]:
lazy_mean = speed.mean("time")
lazy_max = speed.max("time")
lazy_std = speed.std("time")

mean_field, max_field, std_field = dask.compute(lazy_mean, lazy_max, lazy_std)
print("computed together:", mean_field.shape, max_field.shape, std_field.shape)
print(f"peak speed anywhere: {float(max_field.max()):.4f}")

## Reduce data larger than memory

Reductions such as `mean`, `std`, or `max` consume the array chunk by chunk and keep only small partial results, so peak memory stays close to a single chunk rather than the whole dataset. This is why the same reduction works whether the collection is a few megabytes or hundreds of gigabytes, given enough time and disk.

In [ ]:
overall_means = lazy_dataset.mean().compute()
print("domain- and time-averaged value per variable:")
overall_means

## Stream a derived result back to disk

`.binary_engine.to_file(...)` materializes one sub-array (one file) at a time. With the default per-timestep writer, each timestep is computed, written, and released before the next one starts, so a derived dataset larger than memory can be persisted without ever holding it all at once.

The unit of streaming is the *file*, not the byte: each sub-array is fully loaded into memory (a Dask compute) and its file is written whole, in a single pass. There is no partial or append write mode, and each completed file is moved into place atomically with `os.replace`, which protects you from partially-updated, corrupted files even if the process is interrupted, but it also means every individual output file must fit in memory. Size the per-file slices in your write specs getter accordingly. If you need true streaming or partial writes, use one of the other xarray-supported formats such as NetCDF or Zarr.

In [ ]:
speed.binary_engine.to_file(getter.writer, derived_directory)

written = sorted(path.name for path in derived_directory.glob("speed-*.bin"))
print(f"wrote {len(written)} files, one per timestep")
written[:6]

Reading the derived files back and comparing against the in-memory result confirms the streamed write is correct.

In [ ]:
roundtrip = xr.open_mfdataset(
    sorted(derived_directory.glob("speed-*.bin")),
    engine="binfile",
    read_specs_getter=getter.reader,
    chunks={"time": 4},
    parallel=True,
)
xr.testing.assert_allclose(roundtrip["speed"].compute(), speed.compute())
print("round-trip matches the in-memory speed field")
roundtrip

## Where to go next

The scaling behavior here comes from Dask and xarray directly, so their guides apply unchanged:

- [Dask array chunks](https://docs.dask.org/en/stable/array-chunks.html) for choosing chunk sizes
- [Dask scheduling](https://docs.dask.org/en/stable/scheduling.html) for threads, processes, and distributed execution
- [Parallel computing with Dask in xarray](https://docs.xarray.dev/en/stable/user-guide/dask.html) for the xarray-specific patterns

For portable archival of results, prefer `Dataset.to_netcdf(...)` or `Dataset.to_zarr(...)`; reach for raw binary output only when an external tool requires that exact layout.